In [19]:
#### Core script to defines and execute rf runs using functions defined in the following scripts

# config          # configures the model run with input data and defaults (where not directly input)
# spatial_cv      # set up the spatial CV blocks based on fold CSV
# train           # trains model and predicts on test data
# evaluate        # calculates metrics for each fold and overall 
# importance      # calculates feature importance + permutation importance for each fold and overall 

from pathlib import Path
import json
import dataclasses
import pandas as pd

from step_a_config import RunConfig
from step_b_spatial_cv import make_spatial_folds
from step_c_train import train_model
from step_d_evaluate import evaluate
from step_e_importance import get_feature_importance

In [20]:
##### Set-up (doesn't change)

# Get the current working directory
cd = Path.cwd().parent.parent 

# set directories
DATA_DIR = Path(f"{cd}/Data/Clean/Training_data/")
RESULTS_DIR = Path(f"{cd}/Results/RF_models/")
FOLD_DIR = Path(f"{cd}/Data/Fold_assignments/") 

In [21]:
#### DEFINE RUN (MANUAL)

version = 'labor' # capital or labor
unit = 'tonne' # USD or tonne
variable_def = 'rtv' # abs or rtv
type_of_model = 'rf'  # rf, qrf, xgb, or lgbm
sample_weighting_option = 'sqrt_inverse'   # "none", "inverse", or "sqrt_inverse"

name_of_run =  f'{version}_{unit}_{variable_def}_{type_of_model}_{sample_weighting_option}'

In [22]:
##### Configuration (some manual changes)

# set target variable 
numerator = "USD" if version == "capital" else "jobs"
precedent = "rtv_" if variable_def == "rtv" else ""
target_var = f'{precedent}log_{version}_intensity_{numerator}_per_million_{unit}' 
 
# config
model_data_type = "absolute" if variable_def == "abs" else "relative"

fold = FOLD_DIR / f"{version}_folds.csv"
model_data = f"{version}_{model_data_type}_final.csv"

RUNS = [
    RunConfig(
        run_name         = name_of_run,
        target           = target_var,   
        dataset          = model_data,
        fold_assignments = fold,
        model_type       = type_of_model,
        id_cols          = ["PROJ_ID", "country_ID"],
        weighting        = sample_weighting_option
    ),
]

# define which columns are feature columns 
feature_cols = ['rtv_log_SOC', 'rtv_log_average_travel_time_city',
       'rtv_log_average_travel_time_port',
       'rtv_log_growing_season_length_days', 'rtv_log_GDP_pc', 'rtv_log_slope',
       'rtv_log_crop_intensity',
       'rtv_log_USD_production_per_million_HA',
       'rtv_log_tonnes_production_per_million_HA',
       'rtv_log_pop_density_people_per_100_km2',
       'rtv_log_buffalo_density_per_100_km2',
       'rtv_log_chicken_density_per_100_km2',
       'rtv_log_cattle_density_per_100_km2',
       'rtv_log_goats_density_per_100_km2', 'rtv_log_pigs_density_per_100_km2',
       'rtv_log_sheep_density_per_100_km2',
       'rtv_log_livestock_density_LU_per_100_km2',
       'rtv_log_female_share_base_100',
       'rtv_log_cereals_share_base_100_production_USD',
       'rtv_log_fibres_share_base_100_production_USD',
       'rtv_log_fruits_share_base_100_production_USD',
       'rtv_log_oilcrops_share_base_100_production_USD',
       'rtv_log_pulses_share_base_100_production_USD',
       'rtv_log_roots_tubers_share_base_100_production_USD',
       'rtv_log_rest_of_crops_share_base_100_production_USD',
       'rtv_log_sugar_crops_share_base_100_production_USD',
       'rtv_log_vegetables_share_base_100_production_USD',
       'rtv_log_rubber_share_base_100_production_USD',
       'rtv_log_ruminants_share_base_100_production_USD',
       'rtv_log_monogastrics_share_base_100_production_USD',
       'rtv_log_poultry_share_base_100_production_USD',
       'rtv_log_child_dependency_ratio_per_hundred',
       'rtv_probability_survival_land_use_objective',
       'rtv_probability_economic_land_use_objective']

In [23]:
##### Define function to save results into defined directory
def save_results(results, config, out_dir):
    out_dir.mkdir(parents=True, exist_ok=True) # creates folder if it doesn't already exist 

    # save config details for run 
    # convert any Path objects to strings so json can handle them
    config_dict = dataclasses.asdict(config)
    config_dict = {
        k: str(v) if isinstance(v, Path) else v
        for k, v in config_dict.items()
    }
    (out_dir / "config.json").write_text(
        json.dumps(config_dict, indent=2)
    )

    # save predictions across all folds
    results["predictions"].to_parquet(out_dir / "predictions.parquet", index=False)

    # save the best parameters for each fold
    params_df = pd.DataFrame([
        {"fold": r["fold"], **r["best_params"]}
        for r in results["fold_results"]
    ])
    params_df.to_csv(out_dir / "best_params.csv", index=False)

    # save metrics and importance scores
    results["metrics"].to_csv(out_dir / "metrics.csv", index=False)
    results["country_metrics"].to_csv(out_dir / "country_metrics.csv", index=False)
    results["importance"].to_csv(out_dir / "importance.csv", index=False)
    
    print(f"  saved to {out_dir}")

In [24]:
##### RUN   

# define the function to run all scripts in order 
def run(config):
    print(f"\n{'═'*60}")
    print(f"  run: {config.run_name}")
    print(f"  target: {config.target}  |  model: {config.model_type}")
    print(f"{'═'*60}")

    # load model data
    df = pd.read_csv(DATA_DIR / config.dataset)

    # run spatial folds functions 
    print("\n── Spatial folds ────────────────────────────────────────")
    folds = make_spatial_folds(df, config)

    # run training and prediction functions
    print("\n── Training ─────────────────────────────────────────────")
    results = train_model(df, feature_cols, folds, config)

    # run evaluation functions
    results["metrics"], results["country_metrics"] = evaluate(results, config)

    # run importance functions
    print("\n── Feature importance ───────────────────────────────────")
    results["importance"] = get_feature_importance(results, df, config)

    # run save function
    print("\n── Saving ───────────────────────────────────────────────")
    out_dir = RESULTS_DIR / config.run_name
    save_results(results, config, out_dir)

# Run run function for each model configuration defined above
for config in RUNS:
    run(config)


════════════════════════════════════════════════════════════
  run: labor_tonne_rtv_rf_sqrt_inverse
  target: rtv_log_labor_intensity_jobs_per_million_tonne  |  model: rf
════════════════════════════════════════════════════════════

── Spatial folds ────────────────────────────────────────
Excluded 4 rows from countries missing from fold CSV
  fold_1: 32 train countries (5,895 rows) | 1 test countries (5,136 rows)
  fold_2: 32 train countries (9,765 rows) | 1 test countries (1,266 rows)
  fold_3: 32 train countries (8,323 rows) | 1 test countries (2,708 rows)
  fold_4: 30 train countries (1,921 rows) | 3 test countries (9,110 rows)

── Training ─────────────────────────────────────────────

── fold_1 ──────────────────────────────
  test countries: ['BRA']
  weighting: sqrt_inverse (min=0.067, max=2.467)


/Users/carinamanitius/Library/Python/3.9/lib/python/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


  best params: {'n_estimators': 500, 'min_samples_leaf': 2, 'max_features': 0.5, 'max_depth': 20}

── fold_2 ──────────────────────────────
  test countries: ['CAN']
  weighting: sqrt_inverse (min=0.049, max=2.470)


/Users/carinamanitius/Library/Python/3.9/lib/python/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


  best params: {'n_estimators': 500, 'min_samples_leaf': 2, 'max_features': 0.5, 'max_depth': 20}

── fold_3 ──────────────────────────────
  test countries: ['USA']
  weighting: sqrt_inverse (min=0.049, max=2.468)
  best params: {'n_estimators': 500, 'min_samples_leaf': 2, 'max_features': 0.5, 'max_depth': 20}

── fold_4 ──────────────────────────────
  test countries: ['BRA', 'CAN', 'USA']
  weighting: sqrt_inverse (min=0.132, max=2.324)
  best params: {'n_estimators': 500, 'min_samples_leaf': 2, 'max_features': 0.5, 'max_depth': 20}

── Evaluation ───────────────────────────────
   fold  test_rmse  test_mae  test_r2  test_rmse_orig  test_mae_orig  test_med_abs_err  train_rmse  train_mae  train_r2  train_rmse_orig  train_mae_orig  train_med_abs_err
 fold_1   1.174001  0.937882 0.526903       95.542652      10.422130          0.794628    0.297500   0.196174  0.952180        26.487455        2.172618           0.135084
 fold_2   1.138608  0.877992 0.443545       15.167581       3.60073

/Users/carinamanitius/Library/Python/3.9/lib/python/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
/Users/carinamanitius/Library/Python/3.9/lib/python/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
/Users/carinamanitius/Library/Python/3.9/lib/python/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(



── Feature Importance (top 15) ──────────────────────────────
                                          feature  impurity_mean  impurity_std  permutation_mean  permutation_std
         rtv_log_tonnes_production_per_million_HA       0.119734      0.041004          0.104632         0.051412
  rtv_log_ruminants_share_base_100_production_USD       0.062058      0.029376          0.084615         0.038223
           rtv_log_pop_density_people_per_100_km2       0.034761      0.013713          0.055163         0.039539
   rtv_log_oilcrops_share_base_100_production_USD       0.144771      0.058321          0.054407         0.047316
                 rtv_log_average_travel_time_port       0.036942      0.015086          0.032581         0.022406
                           rtv_log_crop_intensity       0.051160      0.026646          0.030349         0.021881
rtv_log_sugar_crops_share_base_100_production_USD       0.054780      0.028908          0.029827         0.029542
         rtv_log_livestoc